# 05 - Detail du feature engineering

Ce notebook reprend la logique du fichier `feature_engineering.py`, mais de facon plus lisible pour comprendre les variables creees.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(BASE_DIR / 'data' / 'customer_churn.csv')
df.head()

## Traitement des plaintes

Les valeurs manquantes de `complaint_type` signifient simplement qu il n y a pas eu de plainte connue. On les remplace donc par `Aucune Plainte`.

In [ ]:
data = df.copy()
data['complaint_type'] = data['complaint_type'].fillna('Aucune Plainte')
data['has_complaint'] = (data['complaint_type'] != 'Aucune Plainte').astype(int)
data[['complaint_type', 'has_complaint']].head()

## Variables metier simples

On cree des indicateurs binaires faciles a expliquer : risque paiement, faible satisfaction, inactivite, contrat mensuel.

In [ ]:
data['payment_risk'] = ((data['payment_failures'] > 0) | (data['price_increase_last_3m'] == 'Yes')).astype(int)
data['low_satisfaction'] = ((data['nps_score'] < 0) | (data['csat_score'] <= 2) | (data['survey_response'] == 'Unsatisfied')).astype(int)
data['inactive_customer'] = ((data['monthly_logins'] <= 5) | (data['last_login_days_ago'] >= 20)).astype(int)
data['monthly_contract'] = (data['contract_type'] == 'Monthly').astype(int)
data[['payment_risk', 'low_satisfaction', 'inactive_customer', 'monthly_contract']].head()

## Ratios et scores

Les ratios permettent de comparer les clients plus justement. Par exemple, un client ancien avec 3 tickets support n est pas dans la meme situation qu un nouveau client avec 3 tickets.

In [ ]:
tenure_safe = data['tenure_months'].clip(lower=1)
logins_safe = data['monthly_logins'].clip(lower=1)

data['tickets_per_tenure'] = data['support_tickets'] / tenure_safe
data['revenue_per_month'] = data['total_revenue'] / tenure_safe
data['fee_per_login'] = data['monthly_fee'] / logins_safe
data['support_pressure'] = data['support_tickets'] * data['avg_resolution_time']
data['engagement_score'] = (
    data['monthly_logins'] * 0.30
    + data['weekly_active_days'] * 2.0
    + data['features_used'] * 1.5
    + data['usage_growth_rate'] * 10
    - data['last_login_days_ago'] * 0.40
)
data['satisfaction_score'] = (
    data['csat_score'] * 10
    + data['nps_score'] * 0.20
    - data['escalations'] * 5
    - data['has_complaint'] * 8
)
data[['tickets_per_tenure', 'fee_per_login', 'engagement_score', 'satisfaction_score']].head()

## Verification rapide

On compare les moyennes selon `churn`. Le but est de voir si les variables creees apportent une separation entre clients qui restent et clients qui partent.

In [ ]:
new_cols = ['has_complaint','payment_risk','low_satisfaction','inactive_customer','monthly_contract','tickets_per_tenure','fee_per_login','support_pressure','engagement_score','satisfaction_score']
data.groupby('churn')[new_cols].mean().round(3)